Bibliotecas Necessárias

In [1]:
import os

import pandas as pd

from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import LabelEncoder

from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report, roc_auc_score

import datetime

import numpy as np

Criando a função que irá registrar os logs do sistema

In [2]:
def registrar_log(texto_log):

  data_atual = datetime.datetime.now().strftime("%d-%m-%Y")

  hora = datetime.datetime.now().strftime("%H-%M")

  with open(f"Logs/logs_{data_atual}.txt", "a", encoding="utf-8") as log:

        # write: Função que tem como objetivo registrar dados nos arquivos.
        log.write(f"{texto_log}\n")
        log.write(f"Hora da execução: {hora}\n")
        log.write("------------------------------------------------------------------------------------------------------------------------------\n")

registrar_log("Inicio da Etapa de Treinamento")

Acessando a pasta de treinamento

In [3]:
try:

    lista_arquivos_treinamento = os.listdir("Treinamento")

    registrar_log("O sistema conseguiu acessar a pasta de treinamento do projeto")

except OSError as erro:

    texto_erro = f"Falha no acesso a pasta de treinamento do projeto {erro}"

    registrar_log(texto_erro)


verificando se não há arquivos diferentes na pasta de treinamento

In [4]:
def verificar_diferentes(lista_arquivos):

    existe_diferente = False

    for arquivo in lista_arquivos:

        if not arquivo.endswith(".csv"):

            texto = f"O arquivo {arquivo} não é um csv de treino ou teste"
        
            registrar_log(texto)

            existe_diferente = True
    
        elif arquivo.endswith(".csv") and arquivo != "train.csv" and arquivo != "test.csv":

            texto = f"a outro arquivo csv presente na pasta. Nome do arquivo: {arquivo} "

            existe_diferente = True

            registrar_log(texto)

    if existe_diferente == False:

           registrar_log("Não há arquivos adicionais ou diferentes na pasta de treinamento")

verificar_diferentes(lista_arquivos_treinamento)
        

Carregando a base de dados de treinamento

In [5]:
try:

    base_treinamento = pd.read_csv(f"Treinamento/train.csv")

    registrar_log("O arquivo de treinamento foi carregado com sucesso")

except FileNotFoundError as erro:

    texto_erro = f"Não foi possivel encontrar ou carregar o erro {erro}"

    registrar_log(texto_erro)

Verificando valores nulos

In [6]:
def verificar_nulos(base_de_dados):

    existe_nulos = False

    for coluna in base_de_dados:

        if base_de_dados[coluna].isnull().sum() > 0:

            texto = f"A coluna {coluna} possui valores nulos."

            existe_nulos = True

            registrar_log(texto)

    
    if existe_nulos == False:

        registrar_log("Não há valores nulos na base de dados")
    
    else:

        registrar_log("A execução do programa foi interrompida")
        
        raise ValueError("Valores nulos foram encontrados na base")


verificar_nulos(base_treinamento)

Renomeando as colunas da base de treino

In [7]:
def renomear_colunas(base_de_dados):

   try:
    
     base_de_dados.rename(columns={
    "Customer_ID":"ID_do_Cliente", 
    "Name":"Nome", 
    "Email":"Email",
    "Account_Age_Days":"Idade_da_conta",
    "Login_Frequency": "Frequencia_login", 
    "Daily_Usage_Mins":"Minutos_uso_diario", 
    "Last_Support_Ticket":"Ultimo_ticket_de_suporte", 
    "Churn": "rotatividade"}, inplace=True)

     texto = f"As colunas da base foram renomeadas"

     registrar_log(texto)

   except KeyError as erro:

    texto = f"Falha na renomeação das colunas da base: {erro}"

    registrar_log(texto)

renomear_colunas(base_treinamento)

                                                        Escolhendo as Caracteristicas Necessárias 

colunas = ID_do_Cliente, Nome, Email, Idade_da_conta, Frequencia_login,
       Minutos_uso_diario, Ultimo_ticket_de_suporte, rotatividade

Caracteristicas Numéricas: Idade_da_conta, Minutos_uso_diario, rotatividade

caracteristicas Categóricas: Frequencia_login (Label Encoder), ultimo_ticket_suporte (OneHotEncoder)

LabelEncoder: Classe que tem como objetivo codificar valores categóricos que POSSUEM uma sequência 
lógica, que é o caso da coluna de frequência_login que possibilita a comparação de frequencia entre
os clientes

OneHotEncoder: Classe que tem como objetivo codificar valores categóricos que NÃO POSSUEM uma
sequência lógica, ou seja, colunas que não possibilitam a comparação dos seus valores


In [8]:
onehot_treinado = None

def aplicar_onehot(base_de_dados):

    global onehot_treinado

    try:

        if onehot_treinado is None:

            onehot_treinado = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

            codificacao = onehot_treinado.fit_transform(base_de_dados[['Ultimo_ticket_de_suporte']])
        
        else:

            codificacao = onehot_treinado.transform(base_de_dados[["Ultimo_ticket_de_suporte"]]) 

        nomes_onehot = onehot_treinado.get_feature_names_out(['Ultimo_ticket_de_suporte'])

        df_one_hot = pd.DataFrame(codificacao, columns=nomes_onehot, index = base_de_dados.index)

        base_de_dados = pd.concat([base_de_dados.drop(columns=['Ultimo_ticket_de_suporte']), df_one_hot], axis=1)

        registrar_log("A coluna Ultimo_ticket_de_suporte foi codificada")

        return base_de_dados
    
    except KeyError as erro:

        texto = f"Falha na codificação da coluna Ultimo_ticket_de_suporte, por favor verifique se a coluna possui o nome padrão definido nas regras de negócios: {erro}"
        
        registrar_log(texto)

base_treinamento = aplicar_onehot(base_treinamento)

In [9]:
labelencoder_treinado = None
def aplicar_labelencoder(base_de_dados):

    global labelencoder_treinado

    try:
        
        if labelencoder_treinado is None:

             labelencoder_treinado = LabelEncoder()

             codificacao = labelencoder_treinado.fit_transform(base_de_dados['Frequencia_login'])
            
        else:

            codificacao = labelencoder_treinado.transform(base_de_dados['Frequencia_login'])

        
        base_de_dados['Frequencia_login'] = codificacao

        registrar_log("A coluna Frequencia_login foi codificada com sucesso")

        return base_de_dados

    except KeyError as erro:

        texto = f"Falha na codificação da coluna Frequencia_login, por favor verifique se a coluna possui o nome padrão definido nas regras de negócios: {erro}"

        registrar_log(texto)

base_treinamento = aplicar_labelencoder(base_treinamento)

Treinamento do modelo

In [10]:
colunas_apagadas = ['ID_do_Cliente', 'Nome', 'Email', 'rotatividade']

x_caracteristicas_treino = base_treinamento.drop(columns=colunas_apagadas)

y_alvo_treino = base_treinamento['rotatividade']

xtrain, xtest, ytrain, ytest = train_test_split(x_caracteristicas_treino, y_alvo_treino, test_size=0.2, random_state=42)

regressao_logistica = LogisticRegression(max_iter=1000, random_state=42)

regressao_logistica.fit(xtrain, ytrain)

registrar_log("O treinamento do modelo foi completo")

Testando a acurácia do modelo

In [11]:
y_pred = regressao_logistica.predict_proba(xtest)[:,1]

curva_roc = roc_auc_score(ytest, y_pred)

texto_roc = f"Curva roc: {round(curva_roc,2)}"

registrar_log("Coletando resultados da eficiência do modelo")

registrar_log(texto_roc)

Realizando a etapa de predição dos dados

In [12]:
registrar_log("Iniciando a predição dos dados")

Acessando a pasta que contém os dados da predição


In [13]:
try:

    lista_arquivos_predicao = os.listdir("Dados")

    registrar_log("A pasta Dados foi acessada pelo sistema")

except OSError as erro:

    registrar_log("Falha no acesso a pasta ")


Verificando se há arquivos diferentes do padrão na pasta de dados

In [14]:
verificar_diferentes(lista_arquivos_predicao)

Carregando a base de dados

In [15]:
try:

    base_predicao = pd.read_csv("Dados/test.csv")

    base_relatorio = pd.read_csv("Dados/test.csv")

    registrar_log("A base de dados de predição foi carregada com sucesso")

except FileNotFoundError as erro:

    texto = f"Arquivo não encontrado, por favor verifique se o arquivo existe na pasta: {erro}"

    registrar_log(texto)

Verificando a existência de valores nulos

In [16]:
verificar_nulos(base_predicao)

Renomeando as colunas

In [17]:
renomear_colunas(base_predicao)

Codificando as variáveis categóricas

In [18]:
base_predicao = aplicar_onehot(base_predicao)

base_predicao = aplicar_labelencoder(base_predicao)

Realizando a predição dos dados

In [19]:

x = base_predicao.drop(columns=colunas_apagadas)

predicao = regressao_logistica.predict_proba(x)

registrar_log("Predição do modelo completa")

Criando a coluna de classificação


In [20]:
prob_cancelamento = predicao[:,1]

base_relatorio["% probabilidade"] = prob_cancelamento

condicoes = [prob_cancelamento < 0.3, (prob_cancelamento >= 0.3) & (prob_cancelamento <=0.7), prob_cancelamento > 0.7]

classificacoes = ["Baixo", "Médio", "Alto"]

base_relatorio["Probabilidade"] = np.select(condicoes, classificacoes, default="Não Classificado")

registrar_log("A coluna de classificação das probabilidades foi criada")

registrar_log("A coluna de porcentagem das probabilidades foi criada")

In [21]:
base_relatorio.head(10)

,Customer_ID,Name,Email,Account_Age_Days,Login_Frequency,Daily_Usage_Mins,Last_Support_Ticket,Churn,% probabilidade,Probabilidade
0,068b54d7-7461-4d1c-885e-f5b43efed384,Alyssa Clark,shawn43@example.com,269,Daily,43,Just checking if my payment went through.,0,0.227517,Baixo
1,5d1e924a-5232-4255-8df9-610e13465816,Dr. David Austin MD,woodlydia@example.com,439,Rarely,9,The UI is too confusing. I can't find the expo...,1,0.853025,Alto
2,c07c0a8a-bac0-47f9-b93d-57d61490a610,Dorothy Rose,naguirre@example.org,18,Rarely,1,I've been waiting for support for 3 days. I'm ...,1,0.856600,Alto
3,cfc0646f-0927-4e8e-b40f-75090340aa0a,Miranda Gilbert,xbauer@example.com,405,Daily,22,Is there a tutorial for the new dashboard feat...,0,0.260981,Baixo
4,67855f87-2552-4273-9aff-1b2060638673,Adam Hayden,austinolivia@example.org,24,Daily,88,"Just upgraded to the Pro plan, excited to use it.",0,0.071993,Baixo
5,14315c24-aa38-40f9-8eaa-bf5f242107d9,Matthew Ruiz,donnalewis@example.org,58,Weekly,50,How do I change my password?,0,0.153093,Baixo
6,592b837b-6df9-4267-8009-b8a7d1c63e01,Edgar Taylor,rodriguezcynthia@example.net,590,Weekly,42,Can I add more seats to my current plan?,1,0.159214,Baixo
7,3cbddf97-c9ff-4701-a979-2d2d341aa8fc,Sabrina Perez,wdavis@example.org,185,Rarely,11,Why did my subscription price increase without...,1,0.806549,Alto
8,013b83f4-0c40-4a53-85f2-e2c165cac055,Shawn Macdonald,ramosmichael@example.com,559,Rarely,2,The UI is too confusing. I can't find the expo...,0,0.865182,Alto
9,7ac4c7f9-94a3-454b-afe5-0f5399679504,Becky Cook,lori99@example.net,88,Daily,101,Customer support was super helpful yesterday. ...,0,0.048161,Baixo


Renomeando as colunas do relatório

In [ ]:
renomear_colunas(base_relatorio)

criando o relatório com as classificações

In [22]:
try:

    agora = datetime.datetime.now().strftime("%d-%m-%Y-%H-%M")

    base_relatorio.to_excel(f"Relatorios/relatorio-{agora}.xlsx", index=False, engine="openpyxl")

    registrar_log("O relatório foi salvo com sucesso, por favor, verifique a pasta de relatórios")

except OSError as erro:

    texto = f"Falha na criação do relatório, tente novamente: {erro}"

    registrar_log(texto)


Transferindo os arquivos ja utilizados na predição para o histórico

In [24]:

try:

    os.replace("Dados/test.csv", f"Historico/test-{agora}.csv")

    registrar_log("O arquivo foi utilizado na predição foi removido para o histórico")

except OSError as erro:

    texto = f"Falha na transferencia do arquivo para o histórico de predições{erro}"

    registrar_log(texto)
